In [1]:
import sys
import os
from dotenv import load_dotenv, find_dotenv
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona
from shapely.geometry import box
import pandas as pd

load_dotenv(find_dotenv())

DATA_PATH = os.getenv("DATA_PATH")
DEWEY_PATH = os.getenv("DEWEY_PATH")

In [2]:
df = pd.read_parquet(os.path.join(DATA_PATH, "processed_data", "dewey_ca_la_county_permits.parquet"))

In [41]:
# jurisdiction by year quality statistics

df['FILE_YEAR'] = df['FILE_DATE'].dt.year
df['MISSING_PERMIT_NUMBER'] = df['PERMIT_NUMBER'].isna()
df['MISSING_PERMIT_DATE'] = (df['PERMIT_DATE'].isna()) & (df['STATUS_NORMALIZED'].isin(['Final', 'Active']))
df['MISSING_FINAL_DATE'] = (df['FINAL_DATE'].isna()) & (df['STATUS_NORMALIZED'].isin(['Final']))
df['MISSING_STATUS'] = df['STATUS_NORMALIZED'].isna()
df['MISSING_ZIPCODE'] = df['ZIPCODE'].isna()
df['MISSING_APN'] = df['APN'].isna()
df['MISSING_STREET'] = df['STREET'].isna()
df['GOOD'] = (~df['MISSING_PERMIT_NUMBER']) & (~df['MISSING_FINAL_DATE']) & (~df['MISSING_STATUS']) & (~df['MISSING_ZIPCODE'] | ~df['MISSING_STREET'] | ~df['MISSING_APN'])
df['BAD'] = ~df['GOOD']

dfg = df.groupby(['STATE', 'JURISDICTION', 'FILE_YEAR']).agg(
    COUNT = ('JURISDICTION', 'count'),
    MISSING_PERMIT_NUMBER = ('MISSING_PERMIT_NUMBER', 'mean'),
    MISSING_PERMIT_DATE = ('MISSING_PERMIT_DATE', 'mean'),
    MISSING_FINAL_DATE = ('MISSING_FINAL_DATE', 'mean'),
    MISSING_STATUS = ('MISSING_STATUS', 'mean'),
    MISSING_ZIPCODE = ('MISSING_ZIPCODE', 'mean'),
    MISSING_APN = ('MISSING_APN', 'mean'),
    MISSING_STREET = ('MISSING_STREET', 'mean'),
    PCT_BAD = ('BAD', 'mean'),
    COUNT_GOOD = ('GOOD', 'sum')
).reset_index()

dfg['JURISDICTION_MEAN_COUNT'] = dfg.groupby(['JURISDICTION'])['COUNT'].transform('mean')

# bad if count is <10% of jurisdiction mean count or if pct_bad > 10%
dfg['BAD'] = (dfg['COUNT'] < dfg['JURISDICTION_MEAN_COUNT'] * 0.1) | (dfg['PCT_BAD'] > 0.1)



In [ ]:
# jursidiction usable data statistics

dfc = dfg.loc[dfg['BAD'] == False].groupby(['STATE', 'JURISDICTION']).agg(
    YEAR_MIN = ('FILE_YEAR', 'min'),
    YEAR_MAX = ('FILE_YEAR', 'max'),
    USABLE_PERMITS = ('COUNT_GOOD', 'sum')
).reset_index()

dfc['PERMITS_PER_YEAR'] = dfc['USABLE_PERMITS'] / (dfc['YEAR_MAX'] - dfc['YEAR_MIN'] + 1)

dfc



,STATE,JURISDICTION,YEAR_MIN,YEAR_MAX,USABLE_PERMITS,PERMITS_PER_YEAR
0,CA,Alhambra,2021.0,2024.0,13034,3258.500000
1,CA,Arcadia,1999.0,2023.0,43777,1751.080000
2,CA,Azusa,2000.0,2022.0,27171,1181.347826
3,CA,Beverly Hills,2000.0,2022.0,141726,6162.000000
4,CA,Burbank,1972.0,2025.0,458897,8498.092593
5,CA,Claremont,1993.0,2023.0,46611,1503.580645
6,CA,Culver City,2016.0,2025.0,62192,6219.200000
7,CA,Downey,2010.0,2025.0,25521,1595.062500
8,CA,El Segundo,2002.0,2022.0,53639,2554.238095
9,CA,Gardena,2000.0,2025.0,60678,2333.769231
